In [ ]:
from pgmpy.inference import VariableElimination

inference = VariableElimination(model)

In [ ]:
result = inference.query(
    variables=["Resident_Sleep"],
    evidence={
        "Is_Night": "Yes",
        "Bedroom_Activity": "None"
    }
)

print(result)

In [ ]:
# metrics calculation 
test_data = test.drop(columns=["timestamp", "Home"], errors="ignore")
targets = [
    "Home_Occupancy",
    "Resident_Activity",
    "Resident_Sleep",
    "Leaving_Home",
    "Returning_Home",
    "Unusual_Activity"
]

results = {}

for target in targets:
    actual = []
    predicted = []

    for _, row in test_data.iterrows():
        evidence = {
            col: row[col]
            for col in evidence_nodes
            if col in row.index and pd.notna(row[col])
        }

        query = inference.query(
            variables=[target],
            evidence=evidence,
            show_progress=False
        )

        predicted_state = query.state_names[target][
            query.values.argmax()
        ]

        actual.append(row[target])
        predicted.append(predicted_state)

    results[target] = (actual, predicted)


In [ ]:
#calculate accuracy
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd

results_table = []

for target, (actual, predicted) in results.items():
    results_table.append({
        "Target": target,
        "Accuracy": accuracy_score(actual, predicted),
        "Precision": precision_score(actual, predicted, average="weighted", zero_division=0),
        "Recall": recall_score(actual, predicted, average="weighted", zero_division=0),
        "F1-Score": f1_score(actual, predicted, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results_table)
print(results_df.round(4))